<a href="https://colab.research.google.com/github/ArmandoArV/IntroDataScienceProyecto/blob/armandoav%2FInitialFileCommit/DataScienceProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis Global de Precios de Combustibles 2020–2026

**Pontificia Universidad Católica de Chile — Introducción a Data Science**

## Estructura del proyecto
```
Datasets/
  raw/          ← Datos originales (nunca se modifican)
  processed/    ← Datos limpios listos para análisis
```

## Referencia de buenas prácticas
Rule, A. et al. (2019). *Ten simple rules for reproducible research in Jupyter Notebooks.* PLoS Comput Biol, 15(7), e1007007.

## 0. Instalación de dependencias

In [ ]:
!pip install -q kagglehub pandas matplotlib seaborn scikit-learn xgboost

## 1. Configuración de credenciales de Kaggle

Para descargar el dataset necesitas tu archivo `kaggle.json`.
- Descárgalo desde: https://www.kaggle.com/settings → API → Create New Token

**En Google Colab:** se te pedirá subir el archivo interactivamente.  
**En local:** coloca `kaggle.json` en `~/.kaggle/` (Linux/Mac) o `C:\Users\<tu_usuario>\.kaggle\` (Windows).

In [ ]:
import os
import json

# Detectar si estamos en Google Colab
IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(globals().get('get_ipython', lambda: None)())

kaggle_dir = os.path.expanduser('~/.kaggle')
kaggle_json = os.path.join(kaggle_dir, 'kaggle.json')

if not os.path.exists(kaggle_json):
    if IN_COLAB:
        from google.colab import files
        print('Sube tu archivo kaggle.json:')
        uploaded = files.upload()
        os.makedirs(kaggle_dir, exist_ok=True)
        with open(kaggle_json, 'wb') as f:
            f.write(uploaded['kaggle.json'])
        os.chmod(kaggle_json, 0o600)
    else:
        raise FileNotFoundError(
            f'Coloca kaggle.json en {kaggle_dir}\n'
            'Descárgalo de: https://www.kaggle.com/settings → API → Create New Token'
        )

with open(kaggle_json) as f:
    creds = json.load(f)
    os.environ['KAGGLE_USERNAME'] = creds['username']
    os.environ['KAGGLE_KEY'] = creds['key']

print(f'✅ Kaggle configurado para: {creds["username"]}')

## 2. Descarga y carga del dataset

**Fuente:** [Global Fuel Prices 2020–2026](https://www.kaggle.com/datasets/belbino/global-fuel-prices-20202026)  
27,468 filas · 84 países · 10 columnas

In [ ]:
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

RAW_PATH = 'Datasets/raw/global_fuel_prices.csv'

if os.path.exists(RAW_PATH):
    print('Cargando datos desde archivo local...')
    df = pd.read_csv(RAW_PATH)
else:
    print('Descargando desde Kaggle...')
    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        'belbino/global-fuel-prices-20202026',
        '',
    )
    # Guardar copia en raw (nunca se modifica)
    os.makedirs('Datasets/raw', exist_ok=True)
    df.to_csv(RAW_PATH, index=False)
    print(f'Guardado en {RAW_PATH}')

print(f'Shape: {df.shape}')
df.head()

## 3. Exploración inicial

In [ ]:
print('=== Info del dataset ===')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
print(f'Países: {df["country"].nunique()}')
print(f'Rango de fechas: {df["date"].min()} → {df["date"].max()}')
print(f'Regiones: {df["region"].unique().tolist()}')
print(f'Niveles de ingreso: {df["income_level"].unique().tolist()}')
print(f'Niveles de subsidio: {df["subsidy_level"].unique().tolist()}')
print()
print('=== Valores nulos ===')
print(df.isnull().sum())
print()
print('=== Estadísticas descriptivas ===')
df.describe()

## 4. Limpieza de datos

Pipeline de limpieza basado en el análisis de calidad:
- **Sin valores nulos** (0 en todas las columnas)
- **Sin duplicados** (0 filas duplicadas, 0 por date+country)
- **Sin valores negativos** en precios ni impuestos
- **Frecuencia semanal** consistente (7 días entre observaciones)
- 26 registros con `tax_percentage = 0` (válidos: países sin impuesto a combustibles)

Transformaciones aplicadas:
1. Conversión de tipos (date → datetime, categóricas → category)
2. Validación de integridad (rangos, duplicados, frecuencia temporal)
3. Enriquecimiento con variables temporales (año, mes, trimestre, periodo geopolítico)
4. Cálculo de variables derivadas (spread diésel-gasolina, ratio crudo-retail, precio real)

### 4.1 Conversión de tipos y validación

In [ ]:
df_clean = df.copy()

# --- Conversión de tipos ---
df_clean['date'] = pd.to_datetime(df_clean['date'])

cat_cols = ['country', 'region', 'income_level', 'subsidy_level']
for col in cat_cols:
    df_clean[col] = df_clean[col].astype('category')

# --- Validación de integridad ---
price_cols = ['petrol_usd_liter', 'diesel_usd_liter', 'lpg_usd_liter', 'brent_crude_usd']
checks = {
    'Valores nulos': df_clean.isnull().sum().sum(),
    'Filas duplicadas': df_clean.duplicated().sum(),
    'Duplicados date+country': df_clean.duplicated(subset=['date', 'country']).sum(),
    'Precios negativos': sum((df_clean[c] < 0).sum() for c in price_cols),
    'Tax negativos': (df_clean['tax_percentage'] < 0).sum(),
}

print('=== Validación de integridad ===')
all_ok = True
for check, count in checks.items():
    status = '✅' if count == 0 else '⚠️'
    if count > 0:
        all_ok = False
    print(f'{status} {check}: {count}')

# Verificar frecuencia temporal
dates_sorted = df_clean['date'].sort_values().unique()
gaps = pd.Series(dates_sorted).diff().dt.days.dropna()
consistent_freq = gaps.min() == gaps.max() == 7
print(f'{"✅" if consistent_freq else "⚠️"} Frecuencia semanal consistente: {consistent_freq} ({int(gaps.min())}–{int(gaps.max())} días)')
print(f'\n✅ Tipos de datos:')
print(df_clean.dtypes)

### 4.2 Variables temporales

In [ ]:
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month
df_clean['quarter'] = df_clean['date'].dt.quarter
df_clean['week'] = df_clean['date'].dt.isocalendar().week.astype(int)

# Periodos geopolíticos clave para el análisis
def assign_period(date):
    if date < pd.Timestamp('2020-03-11'):
        return 'Pre-COVID'
    elif date < pd.Timestamp('2021-06-01'):
        return 'COVID-19'
    elif date < pd.Timestamp('2022-02-24'):
        return 'Recuperación'
    elif date < pd.Timestamp('2023-06-01'):
        return 'Guerra Ucrania'
    else:
        return 'Post-crisis'

df_clean['period'] = df_clean['date'].apply(assign_period).astype('category')

print('Distribución por periodo:')
print(df_clean['period'].value_counts().sort_index())

### 4.3 Variables derivadas

In [ ]:
# Spread diésel vs gasolina (diferencia de precio)
df_clean['diesel_petrol_spread'] = df_clean['diesel_usd_liter'] - df_clean['petrol_usd_liter']

# Ratio crudo/retail: qué tanto del precio Brent se traslada al consumidor
# Brent está en USD/barril, convertir a USD/litro (1 barril = 158.987 litros)
df_clean['brent_usd_liter'] = df_clean['brent_crude_usd'] / 158.987
df_clean['crude_retail_ratio'] = df_clean['brent_usd_liter'] / df_clean['petrol_usd_liter']

# Componente impositivo estimado del precio
df_clean['tax_component_petrol'] = df_clean['petrol_usd_liter'] * (df_clean['tax_percentage'] / 100)
df_clean['price_before_tax_petrol'] = df_clean['petrol_usd_liter'] - df_clean['tax_component_petrol']

print(f'Nuevas columnas: {[c for c in df_clean.columns if c not in df.columns]}')
print(f'\nShape final: {df_clean.shape}')
df_clean.head()

### 4.4 Resumen del dataset limpio

In [ ]:
print('=== Dataset limpio ===')
print(f'Filas: {df_clean.shape[0]:,} | Columnas: {df_clean.shape[1]}')
print(f'Periodo: {df_clean["date"].min().date()} → {df_clean["date"].max().date()}')
print(f'Países: {df_clean["country"].nunique()} | Regiones: {df_clean["region"].nunique()}')
print(f'\nColumnas originales (10): {list(df.columns)}')
print(f'Columnas nuevas ({df_clean.shape[1] - df.shape[1]}): {[c for c in df_clean.columns if c not in df.columns]}')
print()
df_clean.describe()

### 4.5 Guardar datos limpios

In [ ]:
CLEAN_PATH = 'Datasets/processed/fuel_prices_clean.csv'
os.makedirs('Datasets/processed', exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
print(f'✅ Datos limpios guardados en {CLEAN_PATH}')
print(f'   Tamaño: {os.path.getsize(CLEAN_PATH) / 1024:.0f} KB')

## 5. EDA (Análisis Exploratorio) — *próximo paso*

In [ ]:
# TODO: Visualizaciones y análisis exploratorio

## 6. Feature Engineering — *próximo paso*

In [ ]:
# TODO: Creación de variables derivadas

## 7. Modelado — *próximo paso*

In [ ]:
# TODO: Entrenamiento de modelos (XGBoost, etc.)

## 8. Conclusiones — *próximo paso*

In [ ]:
# TODO: Resumen de hallazgos